# FX1 — USD Factor Residual Mean-Reversion Strategy
## NUS FT5010 Final-Term Project — Deliverable Notebook

---

## Strategy Idea

FX1 exploits **mean-reversion in idiosyncratic residuals** of G10 currency pairs.

The core insight: when you remove the common USD factor from each pair's returns
(via rolling PCA + Kalman-tracked loadings), the remaining residuals are stationary
(ADF p-value ≈ 0 on all 7 pairs) and mean-revert predictably.

**Signal pipeline:**
1. Rolling PCA (window=60 bars) → common USD factor
2. Kalman filter → dynamic per-pair loadings
3. OU z-score (window=720 bars) → measures residual deviation
4. Session demean (UTC 0–12 only) → removes time-of-day bias
5. Hawkes process gate → enter only when spread spikes are *decaying*
6. Cross-pair correlation filter → block correlated simultaneous signals
7. Macro guard → block if ≥3 pairs have elevated spreads simultaneously
8. HAR-RV volatility forecast → scale leverage to current vol environment

**Execution:**
- Market orders via OANDA REST API (practice account)
- Stop Loss: z-score extends to 2.2σ
- Take Profit: z-score reverts to 0.5σ
- Max 2 concurrent positions, SGD 200k margin per trade

---

## OANDA Credentials
- **Account ID**: `101-003-38807757-001`
- **API Key**: `bc92c0fdaf6522191c5cca914493a283-bfffc8fb47d90822ad1ba0e5274a3ab8`
- **Environment**: practice (paper trading)

## Live Dashboard
**http://135.235.139.80:8000/app** (running on Azure VM)


---
## 0. Setup

Run from the `fx_oanda/` directory (one level up from this notebook):
```bash
pip install -r requirements_fx.txt
```
The notebook uses `sys.path` to import from the repo root automatically.

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so fx_oanda package is importable
repo_root = Path().resolve().parent.parent  # notebooks/ -> fx_oanda/ -> options/
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings('ignore')  # suppress hmmlearn convergence warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from fx_oanda.config.loader import load_config, ensure_dirs
from fx_oanda.data.fetch_oanda import load_cache
from fx_oanda.strategy.pipeline import build_candidate_signals, build_strategy_state
from fx_oanda.backtest.engine import FX1Backtester
from fx_oanda.backtest.metrics import write_backtest_artifacts

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
                     'axes.grid': True, 'grid.alpha': 0.4})

print('Setup complete.')

---
## 1. Load Configuration and Data

In [ ]:
# Load config (credentials are hardcoded in config/config.yaml)
cfg = load_config()
ensure_dirs(cfg)

print('OANDA Account :', cfg['oanda']['account_id'])
print('Environment   :', cfg['oanda']['environment'])
print('Instruments   :', cfg['instruments'])
print('Granularity   :', cfg['data']['granularity'])

In [ ]:
# Load cached M30 bar data (run 'python run.py -> [1]' first to fetch if not cached)
data = load_cache(cfg)

for pair, df in data.items():
    print(f'{pair}: {len(df):,} bars  |  {df.index[0].date()} → {df.index[-1].date()}')

---
## 2. Build Strategy State

This runs the full signal pipeline (PCA, Kalman, OU z-scores, Hawkes, HMM, HAR-RV).
Takes ~7 minutes on 62k bars — this is a one-time build; the live system caches and refreshes hourly.

In [ ]:
print('Building strategy state (PCA → Kalman → OU → Hawkes → HMM → HAR-RV)...')
print('Expected time: ~7 minutes on 62k bars')

state = build_strategy_state(data, cfg)

train_end = state.train_end_time
n_train   = int(cfg['train_fraction'] * len(state.returns))
n_test    = len(state.returns) - n_train

print(f'\nData split:')
print(f'  Train: {state.returns.index[0].date()} → {train_end.date()}  ({n_train:,} bars, {cfg["train_fraction"]:.0%})')
print(f'  Test : {train_end.date()} → {state.returns.index[-1].date()}  ({n_test:,} bars, {1-cfg["train_fraction"]:.0%})')

### 2a. Stationarity Check (ADF Test on Residuals)

All pairs must have stationary residuals (p-value ≈ 0) for the OU mean-reversion assumption to hold.

In [ ]:
print('ADF p-values on OU residuals (should be ~0 for stationarity):')
print('-' * 40)
for pair, pval in state.diagnostics['adf_pvalues'].items():
    status = '✓ stationary' if pval < 0.05 else '✗ non-stationary'
    print(f'  {pair:<12}  p={pval:.4f}  {status}')

### 2b. HMM Regime Distribution

In [ ]:
counts = state.diagnostics['hmm_state_counts']
total  = sum(counts.values())

print('HMM Regime counts (trained on first 65% of data):')
print('-' * 40)
for regime, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {regime:<15}  {count:>6,} bars  ({count/total:.1%})')

# Plot regime over time (last 2 years of test set)
regime_series = state.regime
test_regime   = regime_series[regime_series.index > train_end].iloc[-17520:]  # last 1yr

fig, ax = plt.subplots(figsize=(14, 2.5))
color_map = {'idiosyncratic': '#00d4aa', 'transitional': '#f59e0b', 'macro': '#ef4444'}
for regime, color in color_map.items():
    mask = test_regime == regime
    ax.fill_between(test_regime.index, 0, mask.astype(int), color=color, alpha=0.7, label=regime)
ax.set_title('HMM Regime Classification — Test Period (last 1 year)')
ax.legend(loc='upper left', fontsize=9)
ax.set_yticks([])
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

### 2c. OU Z-Score Sample (EUR_USD)

In [ ]:
z_test = state.zscores['EUR_USD'][state.zscores.index > train_end].iloc[-8640:]  # last 6 months

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(z_test.index, z_test.values, color='#2196f3', linewidth=0.6, alpha=0.8)
ax.axhline(1.5,  color='#00d4aa', linestyle='--', linewidth=1, label='Entry threshold (1.5σ)')
ax.axhline(-1.5, color='#00d4aa', linestyle='--', linewidth=1)
ax.axhline(2.2,  color='#ef4444', linestyle=':', linewidth=1, label='Stop loss (2.2σ)')
ax.axhline(-2.2, color='#ef4444', linestyle=':', linewidth=1)
ax.set_title('EUR_USD OU Z-Score — Test Period (last 6 months)')
ax.set_ylabel('Z-Score (σ)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

---
## 3. Generate Signals

In [ ]:
# Generate all candidate signals across full history
all_signals = build_candidate_signals(state, cfg)

# Split into train and test
train_signals = all_signals[all_signals.index <= train_end].copy()
test_signals  = all_signals[all_signals.index  > train_end].copy()

print(f'Total candidate signals : {len(all_signals):,}')
print(f'  Train signals         : {len(train_signals):,}')
print(f'  Test signals          : {len(test_signals):,}')
print()
print('Signal distribution by pair (test set):')
print(test_signals['pair'].value_counts().to_string())
print()
print('Signal direction split (test set):')
print(test_signals['direction'].map({1: 'LONG', -1: 'SHORT'}).value_counts().to_string())

In [ ]:
# Show sample signals
test_signals[['pair', 'direction', 'z_score', 'regime', 'har_rv',
              'entry_price', 'sl_price', 'tp_price']].tail(10).round(5)

---
## 4. Walk-Forward Backtest (Out-of-Sample)

The backtester is **event-based**: it iterates signal by signal, tracks active positions,
applies cash-buffer guards, and simulates realistic fills with spread costs.

- **Entry**: market order at next bar's close (assume_fills=True for backtesting)
- **Exit**: TP / SL check bar-by-bar; time-stop after 6 bars (3 hours)
- **No lookahead**: model trained on first 65%, signals generated only on 35% test period

In [ ]:
import math

print('Running event-based backtest on test signals...')
result = FX1Backtester(cfg).run(test_signals, state.prices)
m = result.summary

print(f'\n{'─'*45}')
print(f'  Total Trades    : {m["total_trades"]}')
print(f'  Win Rate        : {m["win_rate"]:.1%}')
print(f'  Profit Factor   : {m["profit_factor"]:.2f}')
print(f'  Net PnL         : SGD {m["net_pnl"]:,.0f}')
print(f'  Total Return    : {m["total_return"]:+.2%}')
print(f'  Sharpe Ratio    : {m["sharpe"]:.2f}')
print(f'  Max Drawdown    : {m["max_drawdown"]:.2%}')
print(f'  Avg Leverage    : {m["avg_leverage"]:.1f}×')
print(f'  Avg Win         : SGD {m["avg_win"]:,.0f}')
print(f'  Avg Loss        : SGD {m["avg_loss"]:,.0f}')
print(f'{'─'*45}')

### 4a. Benchmark Comparison — EUR/USD Buy-and-Hold

In [ ]:
# EUR/USD buy-and-hold over the same test period
eur_prices = state.prices['EUR_USD'][state.prices['EUR_USD'].index > train_end]['close'].dropna()
bnh_return = float(eur_prices.iloc[-1] / eur_prices.iloc[0] - 1.0)
daily_eur  = eur_prices.resample('1D').last().dropna().pct_change().dropna()
bnh_sharpe = float(daily_eur.mean() / daily_eur.std() * 252**0.5) if daily_eur.std() > 0 else 0.0
bnh_dd     = float(((eur_prices - eur_prices.cummax()) / eur_prices.cummax()).min())

print('─' * 55)
print(f'{"Metric":<20}  {"FX1 Strategy":>15}  {"EUR/USD B&H":>12}')
print('─' * 55)
print(f'{"Total Return":<20}  {m["total_return"]:>+14.2%}  {bnh_return:>+11.2%}')
print(f'{"Sharpe Ratio":<20}  {m["sharpe"]:>15.2f}  {bnh_sharpe:>12.2f}')
print(f'{"Max Drawdown":<20}  {m["max_drawdown"]:>14.2%}  {bnh_dd:>11.2%}')
print('─' * 55)

### 4b. Equity Curve

In [ ]:
if not result.equity_curve.empty:
    eq = result.equity_curve.set_index('time')['balance'].sort_index()
    initial = 100_000.0
    drawdown = (eq - eq.cummax()) / eq.cummax()

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]},
                                    sharex=True)

    ax1.plot(eq.index, eq.values, color='#2196f3', linewidth=1.5, label='FX1 Strategy')
    ax1.axhline(initial, color='#6b7280', linestyle='--', linewidth=0.8, label='Initial capital')
    ax1.set_title('FX1 Walk-Forward Equity Curve — Out-of-Sample Test Period', fontsize=13)
    ax1.set_ylabel('Account Balance (SGD)')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'SGD {x:,.0f}'))
    ax1.legend()

    ax2.fill_between(drawdown.index, drawdown.values, 0, color='#ef4444', alpha=0.5)
    ax2.set_ylabel('Drawdown')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

    plt.tight_layout()
    plt.show()
else:
    print('No trades to plot.')

### 4c. P&L Distribution by Pair

In [ ]:
if not result.trades.empty:
    trades = result.trades

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # P&L by pair
    pnl_by_pair = trades.groupby('pair')['pnl'].sum().sort_values()
    colors = ['#ef4444' if v < 0 else '#00d4aa' for v in pnl_by_pair.values]
    axes[0].barh(pnl_by_pair.index, pnl_by_pair.values, color=colors)
    axes[0].set_title('Net PnL by Pair')
    axes[0].set_xlabel('SGD')
    axes[0].axvline(0, color='black', linewidth=0.8)

    # PnL distribution histogram
    wins   = trades[trades['pnl'] >= 0]['pnl']
    losses = trades[trades['pnl'] <  0]['pnl']
    axes[1].hist(losses.values, bins=20, color='#ef444488', edgecolor='#ef4444', label='Losses')
    axes[1].hist(wins.values,   bins=20, color='#00d4aa88', edgecolor='#00d4aa', label='Wins')
    axes[1].axvline(float(trades['pnl'].mean()), color='#f59e0b', linestyle='--',
                    linewidth=1.5, label=f'Expectancy ({trades["pnl"].mean():,.0f} SGD)')
    axes[1].set_title('P&L Distribution')
    axes[1].set_xlabel('SGD per trade')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    # Exit reason breakdown
    print('\nExit reason breakdown:')
    print(trades.groupby('exit_reason')['pnl'].agg(['count', 'mean', 'sum']).round(1).to_string())

---
## 5. Save Backtest Artifacts

In [ ]:
from datetime import datetime, timezone

run_dir = Path('..') / 'backtest' / 'results' / datetime.now(timezone.utc).strftime('notebook_%Y%m%d_%H%M%S')
summary = write_backtest_artifacts(run_dir, result, test_signals, state.diagnostics)
print(f'Artifacts saved to: {run_dir.resolve()}')
print(f'  trades.csv, equity_curve.csv, signals.csv, summary.json')

---
## 6. Live System Overview

The live system is running on Azure VM at **http://135.235.139.80:8000/app**

Architecture:
- `runtime/live_trader.py` — bar loop: runs on every M30 close, rebuilds state hourly
- `backend/api.py` — FastAPI server: serves dashboard, broadcasts WebSocket updates every 10s
- `execution/oanda_exec.py` — OANDA REST wrapper: place/close market orders
- `frontend/` — React+Plotly dashboard with kill switch

The kill switch in the dashboard calls `POST /api/kill` which:
1. Arms the kill switch flag in `artifacts/kill_switch.json`
2. On next bar, `live_trader.py` detects it and calls `executor.flatten_all()`
3. All open positions are closed via OANDA market orders

In [ ]:
# Verify OANDA connection is working
import requests

headers = {
    'Authorization': f'Bearer {cfg["oanda"]["api_key"]}',
    'Content-Type': 'application/json',
}
base_url = cfg['oanda']['base_url']
account_id = cfg['oanda']['account_id']

resp = requests.get(f'{base_url}/v3/accounts/{account_id}/summary', headers=headers)
if resp.status_code == 200:
    acct = resp.json()['account']
    print(f'OANDA connection: OK')
    print(f'  Balance  : {float(acct["balance"]):,.2f} {acct["currency"]}')
    print(f'  NAV      : {float(acct["NAV"]):,.2f} {acct["currency"]}')
    print(f'  Open P&L : {float(acct["unrealizedPL"]):+,.2f}')
    print(f'  Open pos : {acct["openPositionCount"]}')
else:
    print(f'OANDA connection failed: {resp.status_code} — {resp.text[:200]}')